# DenLsNet — Model Evaluation & Explainability Notebook

This notebook loads a saved DenLsNet model from this repository, evaluates it on the test set (accuracy, precision, recall, F1, confusion matrix, ROC/AUC for binary), and runs the implemented explainability pipeline (Grad-CAM, Grad-CAM++, SHAP, LIME) and the quantitative benchmarking (IoU, Insertion/Deletion AUC, Stability).

Notes / assumptions:
- You should run this notebook from the repository root. The notebook will add the repo root to `sys.path`.
- Set the `MODEL_PATH` and (optionally) `DATA_ROOT` variables below if the defaults do not match your environment.
- For speed while experimenting, set `NUM_EXPLAIN_SAMPLES` / `NUM_BENCH_SAMPLES` small (e.g., 5-20).
- If a checkpoint contains `checkpoint['model']` the whole model will be used; otherwise the notebook will attempt to instantiate a model and load `model_state_dict`. If you have a custom checkpoint layout, update the loader cell accordingly.

In [ ]:
# Setup imports and paths
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Make sure repository root is on sys.path (adjust if you open the notebook from a different folder)
REPO_ROOT = Path('..').resolve() if Path('.').name == 'notebooks' else Path('.').resolve()
print(f'REPO_ROOT = {REPO_ROOT}')
sys.path.insert(0, str(REPO_ROOT))

# Helpful device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# User-editable settings: set the model path and data path here
# Example: MODEL_PATH = 'weight/save/40/iaff40_5.pth'
MODEL_PATH = None  # set to your checkpoint path string, or leave None to try to auto-discover
# DATA_ROOT will default to config.valid (config.py) but you can override here
DATA_ROOT = None  # optional: e.g. 'datasets/BreaKHis 400X/test'
# How many samples to run for explainability benchmarking (set low for quick runs)
NUM_BENCH_SAMPLES = 10
NUM_EXPLAIN_SAMPLES = 3

# Auto-discover model if MODEL_PATH is None (search common folders)
if MODEL_PATH is None:
    candidates = list(Path('weight').rglob('*.pth')) + list(Path('weight').rglob('*.pt')) + list(Path('*.pth')) + list(Path('*.pt'))
    candidates += list(Path('weight').rglob('*.pth.tar')) + list(Path('weight').rglob('*.tar'))
    candidates = [str(p) for p in candidates]
    if candidates:
        print('Found candidate checkpoints (first selected if multiple):')
        for c in candidates[:10]:
            print(' -', c)
        MODEL_PATH = candidates[0]
    else:
        print('No checkpoint auto-discovered. Please set MODEL_PATH to your checkpoint file path.')

print('MODEL_PATH =', MODEL_PATH)

In [ ]:
# Robust model loader: handles checkpoints that store 'model' or 'model_state_dict'
import torch
from importlib import import_module
import config

def load_checkpoint_model(checkpoint_path, device=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = None
    # If the checkpoint saved the whole model object (common in this repo)
    if isinstance(checkpoint, dict) and 'model' in checkpoint:
        model = checkpoint['model']
        model.to(device)
        model.eval()
        print('Loaded model object from checkpoint.')
        return model, device

    # If a state_dict is present, attempt to instantiate a model and load the weights
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        print('Checkpoint contains state_dict. Attempting to instantiate model and load weights...')
        # Try common model factories found in repository
        try:
            # Try multiclass factory first
            from model.multiclass_model import create_multiclass_model
            model = create_multiclass_model(num_classes=getattr(config, 'class_num', 2))
            model.load_state_dict(checkpoint['model_state_dict'])
            model.to(device)
            model.eval()
            print('Instantiated MultiClass model and loaded state_dict.')
            return model, device
        except Exception as e_mult:
            print('Multiclass factory failed:', e_mult)
        try:
            from model.model import class_model
            model = class_model()
            model.load_state_dict(checkpoint['model_state_dict'])
            model.to(device)
            model.eval()
            print('Instantiated binary/class_model and loaded state_dict.')
            return model, device
        except Exception as e_bin:
            print('Binary factory failed:', e_bin)

    # If checkpoint is just a model object (not dict) try that
    if not isinstance(checkpoint, dict):
        try:
            model = checkpoint
            model.to(device)
            model.eval()
            print('Checkpoint appears to be a model object (direct load).')
            return model, device
        except Exception as e_obj:
            print('Could not use checkpoint as model object:', e_obj)

    raise RuntimeError('Failed to load model from checkpoint. Please inspect the checkpoint file or update the loader cell.')

# Try to load if MODEL_PATH was discovered
model = None
if MODEL_PATH:
    model, DEVICE = load_checkpoint_model(MODEL_PATH, DEVICE)
else:
    print('No MODEL_PATH set. Set MODEL_PATH and re-run this cell to load the model.')

In [ ]:
# Create a test DataLoader from folder (uses config.valid by default)
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import config

IMG_SIZE = getattr(config, 'img_s', 224)
MEAN = getattr(config, 'dataset_mean', (0.485,0.456,0.406))
STD = getattr(config, 'dataset_std', (0.229,0.224,0.225))
BATCH_SIZE = getattr(config, 'batch_size', 32)
NUM_WORKERS = getattr(config, 'num_workers', 0)

data_root = DATA_ROOT or getattr(config, 'valid', 'datasets/BreaKHis 400X/test')
print('Using data root for test set:', data_root)

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

test_dataset = ImageFolder(data_root, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'Test dataset size: {len(test_dataset)} samples, batches: {len(test_loader)}')
class_names = test_dataset.classes
print('Class names:', class_names)

In [ ]:
# Run evaluation (accuracy, precision, recall, f1, confusion matrix, ROC-AUC for binary)
from evaluation.metrics import ModelEvaluator

evaluator = ModelEvaluator(class_names=class_names)
results = evaluator.evaluate_model(model, test_loader, device=DEVICE, save_results=True, save_dir='notebook_evaluation_results')

# Print a compact summary
metrics = results['metrics']
print('Overall metrics:')
for k in ['accuracy','precision','recall','f1_score']:
    print(f' - {k}:', metrics.get(k))

if 'confusion_matrix' in metrics:
    print('
Confusion matrix:')
    import numpy as np
    print(np.array(metrics['confusion_matrix']))

# Save summary JSON for convenience
import json
with open('notebook_evaluation_results/summary.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved evaluation summary to notebook_evaluation_results/summary.json')

In [ ]:
# Run quantitative explainability benchmarking (IoU, insertion/deletion AUC, stability)
from explainability.quantitative_benchmarking import QuantitativeExplainabilityBenchmark

benchmark = QuantitativeExplainabilityBenchmark(model, device=str(DEVICE), class_names=class_names)
# Use a small number of samples to test; increase NUM_BENCH_SAMPLES for thorough runs
bench_results = benchmark.run_comprehensive_benchmark(test_loader, num_samples=NUM_BENCH_SAMPLES, methods=None, ground_truth_masks=None, save_dir='notebook_explainability_benchmark')

print('
Benchmark summary (per method):')
for method, data in bench_results.items():
    print(method, '-> insertion_auc mean:', data.get('insertion_auc', {}).get('mean'), 'iou mean:', data.get('iou', {}).get('mean'))

In [ ]:
# Generate explanations for a few sample images and visualize (Grad-CAM, Grad-CAM++, SHAP, LIME where available)
import torch.nn.functional as F
from explainability.grad_cam import GradCAM, GradCAMPlusPlus, overlay_heatmap
from explainability.shap_explainer import SHAPExplainer
from explainability.lime_explainer import LIMEExplainer

# Helper: convert tensor -> displayable image (denormalize)
def tensor_to_image(tensor, mean=MEAN, std=STD):
    denorm = tensor.clone()
    for c in range(3):
        denorm[0, c] = denorm[0, c] * std[c] + mean[c]
    img = denorm[0].cpu().numpy().transpose(1,2,0)
    img = np.clip(img, 0, 1)
    return img

# Sample few images from test_loader
samples = []
for images, labels in test_loader:
    for i in range(min(images.size(0), NUM_EXPLAIN_SAMPLES - len(samples))):
        samples.append((images[i:i+1].to(DEVICE), int(labels[i].item())))
    if len(samples) >= NUM_EXPLAIN_SAMPLES:
        break

# Initialize simple explainers for single-sample visualization
# Choose a target layer name heuristically
layer_name_candidates = ['densenet.features.norm5','features.norm5','norm5']
# pick first available target layer from model modules
layer_names = [name for name, _ in model.named_modules()]
target_layer = next((ln for ln in layer_name_candidates if ln in layer_names), None)
if target_layer is None:
    convs = [n for n in layer_names if 'conv' in n.lower()]
    target_layer = convs[-1] if convs else None
print('Using Grad-CAM target layer:', target_layer)

try:
    gradcam = GradCAM(model, target_layer_name=target_layer) if target_layer else None
    gradcam_plus = GradCAMPlusPlus(model, target_layer_name=target_layer) if target_layer else None
    print('Grad-CAM instances ready')
except Exception as e:
    print('Grad-CAM init failed:', e)
    gradcam = gradcam_plus = None

# Optionally initialize SHAP and LIME (may be slow)
shap_explainer = None
lime_explainer = None
try:
    shap_explainer = SHAPExplainer(model, torch.randn(3,3,IMG_SIZE,IMG_SIZE).to(DEVICE), str(DEVICE))
    print('SHAP explainer ready')
except Exception as e:
    print('SHAP init failed:', e)

try:
    lime_explainer = LIMEExplainer(model, str(DEVICE), num_samples=50)
    print('LIME explainer ready')
except Exception as e:
    print('LIME init failed:', e)

# Iterate and visualize
for idx, (img_t, true_label) in enumerate(samples):
    print(f'--- Sample {idx+1} (true label: {true_label})')
    with torch.no_grad():
        out = model(img_t.to(DEVICE))
        probs = F.softmax(out, dim=1)
        pred = int(torch.argmax(probs, dim=1).item())
        conf = float(probs[0, pred].item())
    display_img = tensor_to_image(img_t, mean=MEAN, std=STD)

    explanations = {}
    if gradcam is not None:
        try:
            explanations['gradcam'] = gradcam.generate_cam(img_t, pred)
        except Exception as e:
            print('Grad-CAM failed for sample:', e)
    if gradcam_plus is not None:
        try:
            explanations['gradcam_plus'] = gradcam_plus.generate_cam(img_t, pred)
        except Exception as e:
            print('Grad-CAM++ failed for sample:', e)
    if shap_explainer is not None:
        try:
            shap_vals = shap_explainer.explain_image(img_t, pred)
            if shap_vals is not None:
                heat = np.sum(np.abs(shap_vals), axis=0)
                heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
                explanations['shap'] = heat
        except Exception as e:
            print('SHAP failed for sample:', e)
    if lime_explainer is not None:
        try:
            image_np = img_t[0].cpu().numpy().transpose(1,2,0)
            image_np = np.clip(image_np * STD + MEAN, 0, 1)
            lime_exp, segs = lime_explainer.explain_image(image_np)
            temp, mask = lime_exp.get_image_and_mask(pred, positive_only=False, num_features=10, hide_rest=False)
            maskf = (mask.astype(float) - mask.min()) / (mask.max() - mask.min() + 1e-8)
            explanations['lime'] = maskf
        except Exception as e:
            print('LIME failed for sample:', e)

    # Visualization: original + heatmaps + overlays
    num = 1 + len(explanations)
    cols = min(4, num)
    rows = (num + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = np.array(axes).reshape(-1)
    axes[0].imshow(display_img)
    axes[0].set_title(f'Original (pred={pred}, conf={conf:.2f})')
    axes[0].axis('off')
    i = 1
    for name, heat in explanations.items():
        if i >= len(axes):
            break
        im = axes[i].imshow(heat, cmap='jet')
        axes[i].set_title(name.upper())
        axes[i].axis('off')
        plt.colorbar(im, ax=axes[i], fraction=0.04)
        i += 1
    plt.tight_layout()
    plt.show()

    # Overlay visualizations
    if explanations:
        fig2, ax2 = plt.subplots(1, len(explanations), figsize=(5*len(explanations), 5))
        if len(explanations) == 1:
            ax2 = [ax2]
        j = 0
        for name, heat in explanations.items():
            overlay = overlay_heatmap(display_img, heat, alpha=0.4)
            ax2[j].imshow(overlay)
            ax2[j].set_title(name + ' overlay')
            ax2[j].axis('off')
            j += 1
        plt.tight_layout()
        plt.show()

## Next steps and notes
- If something fails due to missing packages, install the explainability requirements: `pip install -r requirements_explainability.txt` (or the full project requirements).
- If the loader cannot instantiate the model from state_dict, set `MODEL_PATH` and `MODEL_ARCH` (custom) or load the model object directly that was saved inside the checkpoint.
- For a full quantitative run, increase `NUM_BENCH_SAMPLES` to the desired number and provide `ground_truth_masks` if you have pixel-level masks for IoU computation.

Saved artifacts:
- `notebook_evaluation_results/` (evaluation metrics, plots)
- `notebook_explainability_benchmark/` (explainability benchmark JSON, visualizations)